# Tier 2 — Notebook G (v3 + FS): Taiwan Credit Card Default (UCI 350)

**v3 architecture audience-ready notebook.** Same template as the German v3+FS notebook.

**Architecture:**
- **5 base classifiers** (v3 lineup, no skew kernels): RF, GB, XGB, KNN, SVM-RBF.
- **5 meta candidates** (v3 lineup): LR, RF, XGB, MLP, GA-MaxAccLogit.
- **Feature-selection sweep retained** (Full / RFE / Lasso-LR / Forward).

**What gets saved per run:**
- 4 **sub-dataset CSVs** — one per FS variant — for the 16-sub-dataset comparison grid.
- 4 **feature-list CSVs** + FS summary CSV.
- **Per-section results CSVs** — base / classical ensembles / stacker / final combined.

**Dataset:** UCI 350 — Taiwan Credit Card Default (Yeh & Lien 2009; n=30 000, 23 numeric features).

**Highly imbalanced** (78% non-default / 22% default) → SMOTE per fold + GHOST threshold tuning at meta layer. Compared head-to-head against Yeh & Lien (2009) ANN baseline of AUC 0.7716.

Imbalance handling: SMOTE wrapped on every base + meta (`imblearn.Pipeline`, leakage-safe). GHOST tunes the decision threshold at the meta layer.

---

## §0  Environment setup

In [2]:
import os, sys
from pathlib import Path

DRIVE_PROJECT_PATH = '/content/drive/MyDrive/CreditRiskApp'

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    print('Detected Google Colab.')
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path(DRIVE_PROJECT_PATH)
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
except ImportError:
    IN_COLAB = False
    print('Detected local environment.')
    NB_DIR = Path.cwd()
    PROJECT_ROOT = NB_DIR.parents[1] if NB_DIR.name == 'notebooks' else NB_DIR

RESULTS_DIR = PROJECT_ROOT / 'tier2_predictive_models' / 'results'
SUBSETS_DIR = PROJECT_ROOT / 'tier2_predictive_models' / 'fs_subsets'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SUBSETS_DIR.mkdir(parents=True, exist_ok=True)
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'RESULTS_DIR  = {RESULTS_DIR}')
print(f'SUBSETS_DIR  = {SUBSETS_DIR}')

Detected Google Colab.
Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/CreditRiskApp
RESULTS_DIR  = /content/drive/MyDrive/CreditRiskApp/tier2_predictive_models/results
SUBSETS_DIR  = /content/drive/MyDrive/CreditRiskApp/tier2_predictive_models/fs_subsets


### Configuration

In [3]:
DATASET       = 'taiwan'
UCI_ID        = 350
USE_SMOTE     = True  # imbalanced → wrap every base + meta in ImbPipeline(SMOTE → clf)
RANDOM_STATE  = 42
CV_FOLDS      = 3
N_ITER_BAYES  = 12
SMOTE_K       = 5

# Tag prefix for every CSV this notebook saves
RUN_TAG       = 'taiwan_v3'

print(f"Dataset: {DATASET} (UCI id={UCI_ID})")
print(f"USE_SMOTE: {USE_SMOTE}  CV folds: {CV_FOLDS}  Bayes iter: {N_ITER_BAYES}")
print(f"RUN_TAG: {RUN_TAG}")

Dataset: taiwan (UCI id=350)
USE_SMOTE: True  CV folds: 3  Bayes iter: 12
RUN_TAG: taiwan_v3


### Install dependencies

In [4]:
!pip install ucimlrepo xgboost imbalanced-learn ghostml scikit-optimize -q
print('Libraries installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 3.4 MB/s eta 0:00:00
Libraries installed!


### Imports

In [5]:
# --- SSL bypass (strong): for environments with expired CA bundles ---
# Safe to delete this cell once your Python install can verify UCI.
# Bypass 1: default https context (covers urllib/requests defaults).
# Bypass 2: ssl.create_default_context  (covers libraries like
# ucimlrepo that pass cafile=certifi.where() and would otherwise
# ignore the bypass above).
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
_orig_create_default_context = ssl.create_default_context
def _unverified_default_context(*args, **kwargs):
    ctx = _orig_create_default_context(*args, **kwargs)
    ctx.check_hostname = False
    ctx.verify_mode   = ssl.CERT_NONE
    return ctx
ssl.create_default_context = _unverified_default_context
print('SSL verification bypassed for this session (strong).')


SSL verification bypassed for this session (strong).


In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from ucimlrepo import fetch_ucirepo

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, cohen_kappa_score
from sklearn.base import BaseEstimator, ClassifierMixin

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.feature_selection import RFE, SelectFromModel, SequentialFeatureSelector
from xgboost import XGBClassifier
import ghostml

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from skopt import BayesSearchCV
from skopt.space import Real, Integer

print('Imports done.')

Imports done.


### Custom class — `MaxAccLogit`

Genetic-algorithm-tuned logistic regression. Used as one of the 5 meta candidates in the Stacker.
**No skew kernel classes here** — this is the v3 architecture without `SkewPNN` / `SkewSVM`.

In [7]:
class MaxAccLogit(BaseEstimator, ClassifierMixin):
    def __init__(self, lambda_reg=0.01, pop_size=None, n_gen=300,
                 p_crossover=0.8, p_mutation=0.1, elite_frac=0.05,
                 bound=3.0, early_stop=50, scoring='f1_macro', random_state=42):
        self.lambda_reg, self.pop_size, self.n_gen = lambda_reg, pop_size, n_gen
        self.p_crossover, self.p_mutation, self.elite_frac = p_crossover, p_mutation, elite_frac
        self.bound, self.early_stop, self.scoring = bound, early_stop, scoring
        self.random_state = random_state
    def _sigmoid(self, z): return 1.0 / (1.0 + np.exp(-np.clip(z, -50, 50)))
    def _predict_with(self, theta, X): return self._sigmoid(X @ theta[1:] + theta[0])
    def _fitness(self, theta, X, y):
        prob = self._predict_with(theta, X); pred = (prob >= 0.5).astype(int)
        s = f1_score(y, pred, average='macro') if self.scoring == 'f1_macro' else accuracy_score(y, pred)
        return s - self.lambda_reg * np.abs(theta[1:]).sum()
    def _soft_threshold(self, theta):
        b0, b = theta[0], theta[1:]
        b = np.sign(b) * np.maximum(np.abs(b) - self.lambda_reg, 0.0)
        return np.concatenate([[b0], b])
    def fit(self, X, y):
        rng = np.random.default_rng(self.random_state)
        X = np.asarray(X, dtype=float); y = np.asarray(y, dtype=int)
        n, p = X.shape; chrom_len = p + 1
        pop_size = self.pop_size if self.pop_size else 10 * chrom_len
        n_elite = max(1, int(self.elite_frac * pop_size))
        pop = rng.uniform(-self.bound, self.bound, size=(pop_size, chrom_len))
        for i in range(pop_size): pop[i] = self._soft_threshold(pop[i])
        fit_arr = np.array([self._fitness(t, X, y) for t in pop])
        best_history, no_improve = [], 0
        for gen in range(self.n_gen):
            f_min, f_max, f_mean = fit_arr.min(), fit_arr.max(), fit_arr.mean()
            C = 2.0; denom = f_max - f_mean if f_max != f_mean else 1e-10
            if f_min > (C * f_mean - f_max) / max(C - 1, 1e-10):
                a = f_mean * (C - 1) / denom
                b = f_mean * (f_max - C * f_mean) / denom
            else:
                d = f_mean - f_min if f_mean != f_min else 1e-10
                a = f_mean / d; b = -f_mean * f_min / d
            scaled = np.maximum(a * fit_arr + b, 1e-10)
            probs = scaled / scaled.sum()
            sel = rng.choice(pop_size, size=pop_size, p=probs)
            mating = pop[sel].copy()
            for i in range(0, pop_size - 1, 2):
                if rng.random() < self.p_crossover:
                    w = rng.uniform(0, 1, size=chrom_len)
                    p1, p2 = mating[i].copy(), mating[i+1].copy()
                    mating[i]   = w * p1 + (1 - w) * p2
                    mating[i+1] = (1 - w) * p1 + w * p2
            for i in range(pop_size):
                if rng.random() < self.p_mutation:
                    j = rng.integers(0, chrom_len)
                    mating[i, j] = rng.uniform(-self.bound, self.bound)
            for i in range(pop_size):
                mating[i] = self._soft_threshold(mating[i])
            new_fit = np.array([self._fitness(t, X, y) for t in mating])
            elite_idx_old = np.argsort(fit_arr)[-n_elite:]
            worst_idx_new = np.argsort(new_fit)[:n_elite]
            mating[worst_idx_new]  = pop[elite_idx_old]
            new_fit[worst_idx_new] = fit_arr[elite_idx_old]
            pop, fit_arr = mating, new_fit
            best = fit_arr.max(); best_history.append(best)
            if len(best_history) > 1 and best <= best_history[-2] + 1e-8:
                no_improve += 1
                if no_improve >= self.early_stop: break
            else:
                no_improve = 0
        best_idx = np.argmax(fit_arr)
        self.theta_, self.fitness_ = pop[best_idx], fit_arr[best_idx]
        self.classes_ = np.array([0, 1])
        self.n_features_in_ = p
        return self
    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        prob = self._predict_with(self.theta_, X)
        return np.column_stack([1 - prob, prob])
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

print('MaxAccLogit class ready.')

MaxAccLogit class ready.


### `wrap_smote()` helper — applied conditionally
`wrap_smote(model)` returns `ImbPipeline(SMOTE → model)`. Used everywhere when `USE_SMOTE=True`; otherwise the model is used as-is.

In [8]:
def wrap_smote(model, k=SMOTE_K, rs=RANDOM_STATE):
    """ImbPipeline(SMOTE -> clf). SMOTE fires only on .fit(), not on .predict()."""
    return ImbPipeline([
        ('smote', SMOTE(k_neighbors=k, random_state=rs)),
        ('clf',   model),
    ])

def maybe_smote(model):
    """Wrap in SMOTE if USE_SMOTE else pass through."""
    return wrap_smote(model) if USE_SMOTE else model

print(f'wrap_smote() ready.  USE_SMOTE = {USE_SMOTE}')

wrap_smote() ready.  USE_SMOTE = True


### Tiny CSV-save helper
Every CSV save goes through this so we have one place to also trigger a browser download in Colab.

In [9]:
def save_csv(df, path, also_download=True):
    """Save df to `path` and (optionally, only on Colab) trigger a browser download."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f'  Saved → {path}  ({len(df)} rows, {df.shape[1]} cols)')
    if also_download and IN_COLAB:
        try:
            from google.colab import files
            files.download(str(path))
        except Exception as e:
            print(f'  (skipped browser download: {e.__class__.__name__})')

print('save_csv() ready.')

save_csv() ready.


## §1  Data preprocessing

Every step is in its own cell so anything that breaks can be inspected in isolation.

### Step 1 — Load

In [10]:
# --- Prefer a local cached copy; fall back to UCI fetch + save ---
from pathlib import Path

LOCAL_CACHE = Path('taiwan_credit_uci350.csv')

if LOCAL_CACHE.exists():
    print(f'Loading from local cache: {LOCAL_CACHE.resolve()}')
    cached = pd.read_csv(LOCAL_CACHE)
    target_col = 'Y' if 'Y' in cached.columns else cached.columns[-1]
    X_raw = cached.drop(columns=[target_col]).copy()
    y_raw = cached[[target_col]].copy()
else:
    print(f'No local cache; fetching from UCI (id={UCI_ID})...')
    ds    = fetch_ucirepo(id=UCI_ID)
    X_raw = ds.data.features.copy()
    y_raw = ds.data.targets.copy()
    pd.concat([X_raw, y_raw], axis=1).to_csv(LOCAL_CACHE, index=False)
    print(f'Saved local cache: {LOCAL_CACHE.resolve()}')

print(f"Raw shape: X = {X_raw.shape},  y = {y_raw.shape}")
print('First 5 rows of X:')
X_raw.head()


No local cache; fetching from UCI (id=350)...
Saved local cache: /content/taiwan_credit_uci350.csv
Raw shape: X = (30000, 23),  y = (30000, 1)
First 5 rows of X:


,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,...,X14,X15,X16,X17,X18,X19,X20,X21,X22,X23
0,20000,2,2,1,24,2,2,-1,-1,-2,...,689,0,0,0,0,689,0,0,0,0
1,120000,2,2,2,26,-1,2,0,0,0,...,2682,3272,3455,3261,0,1000,1000,1000,0,2000
2,90000,2,2,2,34,0,0,0,0,0,...,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000
3,50000,2,2,1,37,0,0,0,0,0,...,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000
4,50000,1,2,1,57,-1,0,-1,0,0,...,35835,20940,19146,19131,2000,36681,10000,9000,689,679


### Step 2 — Encode target + check class balance

In [11]:
y_flat = y_raw.iloc[:, 0].astype(int).values  # UCI 350: 1 = default (literature convention)
pos_rate = y_flat.mean()
imbalance = abs(pos_rate - 0.5)
is_imbalanced = imbalance > 0.10
print(f'Positives (1=default): {y_flat.sum()}/{len(y_flat)} = {pos_rate:.2%}')
print(f'Imbalance |pos − 0.5| = {imbalance:.2%}')
print(f'is_imbalanced = {is_imbalanced}  →  SMOTE: {"ON" if USE_SMOTE else "OFF"}')

Positives (1=default): 6636/30000 = 22.12%
Imbalance |pos − 0.5| = 27.88%
is_imbalanced = True  →  SMOTE: ON


### Step 3 — Missing-value handling (Taiwan is fully numeric)

In [12]:
print(f'Numeric columns:     {X_raw.shape[1]}')
print(f'Categorical columns: 0  (all features are numeric / ordinal)')
print(f'Total missing values before fill: {X_raw.isna().sum().sum()}')
for c in X_raw.columns:
    if X_raw[c].isna().any():
        X_raw[c] = X_raw[c].fillna(X_raw[c].median())
print(f'Total missing values after fill:  {X_raw.isna().sum().sum()}')

Numeric columns:     23
Categorical columns: 0  (all features are numeric / ordinal)
Total missing values before fill: 0
Total missing values after fill:  0


### Step 4 — No one-hot encoding (Taiwan is fully numeric)

In [13]:
X_enc = X_raw.copy()
print(f'No categorical encoding needed. Columns: {X_enc.shape[1]}')

No categorical encoding needed. Columns: 23


### Step 5 — Standard-scale all features

In [14]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_enc.astype(float)), columns=X_enc.columns)
print(f'After scaling: mean={X_scaled.values.mean():.4f}, std={X_scaled.values.std():.4f}')
feat_names = X_scaled.columns.tolist()
print(f'Total features after preprocessing: {len(feat_names)}')

After scaling: mean=0.0000, std=1.0000
Total features after preprocessing: 23


### Step 6 — Stratified 80/20 train/test split

In [15]:
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X_scaled, y_flat, test_size=0.2, random_state=RANDOM_STATE, stratify=y_flat
)
X_train = X_train_df.values
X_test  = X_test_df.values
print(f'X_train: {X_train.shape}   y_train pos rate: {y_train.mean():.2%}')
print(f'X_test:  {X_test.shape}    y_test  pos rate: {y_test.mean():.2%}')
print(f'Total features: {len(feat_names)}')

X_train: (24000, 23)   y_train pos rate: 22.12%
X_test:  (6000, 23)    y_test  pos rate: 22.12%
Total features: 23


## §2  Feature-selection sweep — produces 4 sub-datasets

For every FS variant we:
1. Compute the selected feature indices.
2. Print the **full list** of selected features.
3. Build the FS-restricted (X_train, X_test) subset.
4. Save the subset as **one CSV** with a `split` column (`train` / `test`) and the target column appended.
5. Save the feature list as a separate CSV.

After this section we have **4 sub-datasets for Taiwan Credit Card Default**. Combined with the other 3 UCI datasets, that's **16 sub-datasets total**.

### Helper — `materialize_subset()`

In [16]:
def materialize_subset(fs_name, fs_idx):
    cols = [feat_names[i] for i in fs_idx]
    print(f'--- FS = {fs_name}  ({len(cols)} features) ---')
    print('Selected feature names:')
    for j, c in enumerate(cols, 1):
        print(f'  {j:2d}. {c}')

    Xtr_df = X_train_df.iloc[:, fs_idx].copy()
    Xte_df = X_test_df.iloc[:, fs_idx].copy()
    Xtr_df['_split'] = 'train';  Xtr_df['_target'] = y_train
    Xte_df['_split'] = 'test';   Xte_df['_target'] = y_test
    subset_df = pd.concat([Xtr_df, Xte_df], axis=0, ignore_index=True)

    subset_path = SUBSETS_DIR / f'{RUN_TAG}_FS_{fs_name.replace("-", "_")}.csv'
    save_csv(subset_df, subset_path, also_download=False)

    feats_df = pd.DataFrame({'idx': fs_idx, 'feature': cols})
    feats_path = SUBSETS_DIR / f'{RUN_TAG}_FS_{fs_name.replace("-", "_")}_features.csv'
    save_csv(feats_df, feats_path, also_download=False)

    return cols, subset_path, feats_path

print('materialize_subset() ready.')

materialize_subset() ready.


### FS variant 1 — Full (all features)

In [17]:
def fs_full(X, y): return np.arange(X.shape[1])

idx_full = fs_full(X_train, y_train)
cols_full, sub_full, feat_full = materialize_subset('Full', idx_full)

--- FS = Full  (23 features) ---
Selected feature names:
   1. X1
   2. X2
   3. X3
   4. X4
   5. X5
   6. X6
   7. X7
   8. X8
   9. X9
  10. X10
  11. X11
  12. X12
  13. X13
  14. X14
  15. X15
  16. X16
  17. X17
  18. X18
  19. X19
  20. X20
  21. X21
  22. X22
  23. X23
  Saved → /content/drive/MyDrive/CreditRiskApp/tier2_predictive_models/fs_subsets/taiwan_v3_FS_Full.csv  (30000 rows, 25 cols)
  Saved → /content/drive/MyDrive/CreditRiskApp/tier2_predictive_models/fs_subsets/taiwan_v3_FS_Full_features.csv  (23 rows, 2 cols)


### FS variant 2 — RFE (RandomForest-wrapped)

In [18]:
def fs_rfe(X, y):
    rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
    rfe = RFE(rf); rfe.fit(X, y)
    return np.where(rfe.support_)[0]

idx_rfe = fs_rfe(X_train, y_train)
cols_rfe, sub_rfe, feat_rfe = materialize_subset('RFE', idx_rfe)

--- FS = RFE  (11 features) ---
Selected feature names:
   1. X1
   2. X5
   3. X6
   4. X12
   5. X13
   6. X14
   7. X15
   8. X16
   9. X17
  10. X18
  11. X19
  Saved → /content/drive/MyDrive/CreditRiskApp/tier2_predictive_models/fs_subsets/taiwan_v3_FS_RFE.csv  (30000 rows, 13 cols)
  Saved → /content/drive/MyDrive/CreditRiskApp/tier2_predictive_models/fs_subsets/taiwan_v3_FS_RFE_features.csv  (11 rows, 2 cols)


### FS variant 3 — Lasso-LR (L1 logistic regression)

In [19]:
def fs_lasso_lr(X, y):
    lasso = LogisticRegression(penalty='l1', solver='liblinear', random_state=RANDOM_STATE)
    sel = SelectFromModel(lasso); sel.fit(X, y)
    return np.where(sel.get_support())[0]

idx_lasso = fs_lasso_lr(X_train, y_train)
cols_lasso, sub_lasso, feat_lasso = materialize_subset('Lasso-LR', idx_lasso)

--- FS = Lasso-LR  (21 features) ---
Selected feature names:
   1. X1
   2. X2
   3. X3
   4. X4
   5. X5
   6. X6
   7. X7
   8. X8
   9. X9
  10. X11
  11. X12
  12. X13
  13. X14
  14. X15
  15. X16
  16. X18
  17. X19
  18. X20
  19. X21
  20. X22
  21. X23
  Saved → /content/drive/MyDrive/CreditRiskApp/tier2_predictive_models/fs_subsets/taiwan_v3_FS_Lasso_LR.csv  (30000 rows, 23 cols)
  Saved → /content/drive/MyDrive/CreditRiskApp/tier2_predictive_models/fs_subsets/taiwan_v3_FS_Lasso_LR_features.csv  (21 rows, 2 cols)


### FS variant 4 — Forward (Sequential, LR estimator)

In [20]:
def fs_forward(X, y):
    lr = LogisticRegression(solver='newton-cg', max_iter=500, random_state=RANDOM_STATE)
    sel = SequentialFeatureSelector(lr, direction='forward', n_jobs=-1)
    sel.fit(X, y)
    return np.where(sel.support_)[0]

idx_forward = fs_forward(X_train, y_train)
cols_forward, sub_forward, feat_forward = materialize_subset('Forward', idx_forward)

--- FS = Forward  (11 features) ---
Selected feature names:
   1. X2
   2. X3
   3. X4
   4. X6
   5. X16
   6. X18
   7. X19
   8. X20
   9. X21
  10. X22
  11. X23
  Saved → /content/drive/MyDrive/CreditRiskApp/tier2_predictive_models/fs_subsets/taiwan_v3_FS_Forward.csv  (30000 rows, 13 cols)
  Saved → /content/drive/MyDrive/CreditRiskApp/tier2_predictive_models/fs_subsets/taiwan_v3_FS_Forward_features.csv  (11 rows, 2 cols)


### Summary — 4 sub-datasets ready

In [21]:
fs_methods = {
    'Full'     : idx_full,
    'RFE'      : idx_rfe,
    'Lasso-LR' : idx_lasso,
    'Forward'  : idx_forward,
}
fs_summary = pd.DataFrame([
    {'FS': name, 'n_features': len(idx), 'features': ', '.join([feat_names[i] for i in idx])}
    for name, idx in fs_methods.items()
])
print('=' * 70); print('  FS sweep summary'); print('=' * 70)
print(fs_summary[['FS', 'n_features']].to_string(index=False))
print('=' * 70)
save_csv(fs_summary, SUBSETS_DIR / f'{RUN_TAG}_FS_summary.csv', also_download=False)

  FS sweep summary
      FS  n_features
    Full          23
     RFE          11
Lasso-LR          21
 Forward          11
  Saved → /content/drive/MyDrive/CreditRiskApp/tier2_predictive_models/fs_subsets/taiwan_v3_FS_summary.csv  (4 rows, 3 cols)


---

### Evaluation helpers

In [22]:
GHOST_THR = np.round(np.arange(0.05, 0.95, 0.01), 2)
USE_GHOST = USE_SMOTE

def evaluate(model, X_tr, X_te, y_tr, y_te):
    """Fit on (X_tr, y_tr); predict on X_te; return Acc / F1 / F1-macro / AUC / thr."""
    model.fit(X_tr, y_tr)
    proba_test = model.predict_proba(X_te)[:, 1]
    if USE_GHOST:
        proba_train = model.predict_proba(X_tr)[:, 1]
        thr = ghostml.optimize_threshold_from_predictions(
            y_tr, proba_train, GHOST_THR, ThOpt_metrics='Kappa')
    else:
        thr = 0.5
    pred = (proba_test >= thr).astype(int)
    return dict(
        acc      = accuracy_score(y_te, pred),
        f1       = f1_score(y_te, pred),
        f1_macro = f1_score(y_te, pred, average='macro'),
        auc      = roc_auc_score(y_te, proba_test),
        thr      = thr,
    )

def sweep_single_model(model_factory, name, category):
    """Run one model_factory across all 4 FS variants."""
    rows = []
    for fs_name, fs_idx in fs_methods.items():
        Xtr = X_train[:, fs_idx]; Xte = X_test[:, fs_idx]
        r = evaluate(model_factory(), Xtr, Xte, y_train, y_test)
        r.update(model=name, category=category, fs=fs_name, n_features=len(fs_idx))
        rows.append(r)
        print(f'  [{category}] {name:<14s} | FS={fs_name:<8s} | '
              f'F1-macro={r["f1_macro"]:.4f}  Acc={r["acc"]:.4f}  F1={r["f1"]:.4f}  thr={r["thr"]:.2f}')
    return rows

print(f'Helpers ready.  USE_GHOST = {USE_GHOST}')

Helpers ready.  USE_GHOST = True


## §3  Base models — simple classifiers

| Model | What it is |
|---|---|
| LogisticRegression | Linear classifier, L2 regularisation. |
| KNN (k=25, Manhattan) | k-nearest neighbours with Manhattan distance. |
| SVM (RBF) | Support vector machine with radial basis function kernel. |
| DecisionTree | Single tree, `max_depth=5`. |

### §3.1 — LogisticRegression × 4 FS

In [23]:
base_lr = sweep_single_model(
    lambda: maybe_smote(LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE)),
    name='LogisticReg', category='Base'
)

  [Base] LogisticReg    | FS=Full     | F1-macro=0.6862  Acc=0.8048  F1=0.4933  thr=0.63
  [Base] LogisticReg    | FS=RFE      | F1-macro=0.6785  Acc=0.7913  F1=0.4881  thr=0.62
  [Base] LogisticReg    | FS=Lasso-LR | F1-macro=0.6865  Acc=0.8053  F1=0.4935  thr=0.63
  [Base] LogisticReg    | FS=Forward  | F1-macro=0.6807  Acc=0.7858  F1=0.4975  thr=0.60


### §3.2 — KNN × 4 FS

In [24]:
base_knn = sweep_single_model(
    lambda: maybe_smote(KNeighborsClassifier(n_neighbors=25, metric='manhattan')),
    name='KNN', category='Base'
)

  [Base] KNN            | FS=Full     | F1-macro=0.6699  Acc=0.7750  F1=0.4836  thr=0.65
  [Base] KNN            | FS=RFE      | F1-macro=0.6615  Acc=0.7773  F1=0.4635  thr=0.69
  [Base] KNN            | FS=Lasso-LR | F1-macro=0.6707  Acc=0.7733  F1=0.4868  thr=0.65
  [Base] KNN            | FS=Forward  | F1-macro=0.6742  Acc=0.7918  F1=0.4785  thr=0.69


### §3.3 — SVM (RBF) × 4 FS

In [25]:
base_svm = sweep_single_model(
    lambda: maybe_smote(SVC(C=1.0, kernel='rbf', gamma='scale',
                            probability=True, random_state=RANDOM_STATE)),
    name='SVM-RBF', category='Base'
)

  [Base] SVM-RBF        | FS=Full     | F1-macro=0.6880  Acc=0.8013  F1=0.5000  thr=0.77
  [Base] SVM-RBF        | FS=RFE      | F1-macro=0.6795  Acc=0.7950  F1=0.4871  thr=0.72
  [Base] SVM-RBF        | FS=Lasso-LR | F1-macro=0.6967  Acc=0.8028  F1=0.5173  thr=0.75
  [Base] SVM-RBF        | FS=Forward  | F1-macro=0.6816  Acc=0.7892  F1=0.4966  thr=0.66


### §3.4 — DecisionTree × 4 FS

In [26]:
base_dt = sweep_single_model(
    lambda: maybe_smote(DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)),
    name='DecisionTree', category='Base'
)

  [Base] DecisionTree   | FS=Full     | F1-macro=0.6847  Acc=0.7973  F1=0.4963  thr=0.61
  [Base] DecisionTree   | FS=RFE      | F1-macro=0.6819  Acc=0.8087  F1=0.4810  thr=0.61
  [Base] DecisionTree   | FS=Lasso-LR | F1-macro=0.6895  Acc=0.8057  F1=0.4996  thr=0.61
  [Base] DecisionTree   | FS=Forward  | F1-macro=0.6793  Acc=0.7840  F1=0.4961  thr=0.58


### §3.5 — Base-model results table + save CSV

In [27]:
base_results = base_lr + base_knn + base_svm + base_dt
df_base = pd.DataFrame(base_results)[
    ['category', 'model', 'fs', 'n_features', 'acc', 'f1', 'f1_macro', 'auc', 'thr']
]
df_base.columns = ['Category', 'Model', 'FS', 'n_feat', 'Accuracy', 'F1', 'F1-macro', 'AUC', 'thr']
for c in ['Accuracy', 'F1', 'F1-macro', 'AUC']: df_base[c] = df_base[c].round(4)
df_base['thr'] = df_base['thr'].round(2)
print('=' * 88); print('  §3 BASE MODELS — 4 classifiers × 4 FS = 16 rows'); print('=' * 88)
print(df_base.to_string(index=False))
save_csv(df_base, RESULTS_DIR / f'{RUN_TAG}_base_results.csv')

  §3 BASE MODELS — 4 classifiers × 4 FS = 16 rows
Category        Model       FS  n_feat  Accuracy     F1  F1-macro    AUC  thr
    Base  LogisticReg     Full      23    0.8048 0.4933    0.6862 0.7105 0.63
    Base  LogisticReg      RFE      11    0.7913 0.4881    0.6785 0.7066 0.62
    Base  LogisticReg Lasso-LR      21    0.8053 0.4935    0.6865 0.7108 0.63
    Base  LogisticReg  Forward      11    0.7858 0.4975    0.6807 0.7006 0.60
    Base          KNN     Full      23    0.7750 0.4836    0.6699 0.7319 0.65
    Base          KNN      RFE      11    0.7773 0.4635    0.6615 0.7264 0.69
    Base          KNN Lasso-LR      21    0.7733 0.4868    0.6707 0.7327 0.65
    Base          KNN  Forward      11    0.7918 0.4785    0.6742 0.7238 0.69
    Base      SVM-RBF     Full      23    0.8013 0.5000    0.6880 0.7535 0.77
    Base      SVM-RBF      RFE      11    0.7950 0.4871    0.6795 0.7458 0.72
    Base      SVM-RBF Lasso-LR      21    0.8028 0.5173    0.6967 0.7529 0.75
    Base      

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## §5  Ensemble models  (v3 architecture — no skew kernels)

### 5a — Classical ensembles (RF, GB, XGB)
RF and XGB are Bayesian-tuned (12 iter × 3-fold CV, scored on `f1_macro`); GB uses sensible defaults.

### 5b — The Stacker (v3)
**5 base classifiers**: RF, GB, XGB, KNN, SVM-RBF.
**5 meta candidates**: LR, RF, XGB, MLP, GA-MaxAccLogit.

### §5a-1 — Bayesian tuning helpers for RF and XGB

In [28]:
def tune_rf(X, y):
    s = BayesSearchCV(
        wrap_smote(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
        {'clf__n_estimators': Integer(100, 300),
         'clf__max_depth': Integer(3, 15),
         'clf__min_samples_split': Integer(2, 10),
         'clf__max_features': Real(0.3, 0.9)},
        n_iter=N_ITER_BAYES, cv=CV_FOLDS, scoring='f1_macro',
        n_jobs=-1, random_state=RANDOM_STATE)
    s.fit(X, y); return s.best_estimator_

def tune_xgb(X, y):
    s = BayesSearchCV(
        wrap_smote(XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0)),
        {'clf__n_estimators': Integer(100, 300),
         'clf__learning_rate': Real(0.02, 0.3, prior='log-uniform'),
         'clf__max_depth': Integer(3, 8),
         'clf__subsample': Real(0.6, 1.0),
         'clf__colsample_bytree': Real(0.6, 1.0)},
        n_iter=N_ITER_BAYES, cv=CV_FOLDS, scoring='f1_macro',
        n_jobs=-1, random_state=RANDOM_STATE)
    s.fit(X, y); return s.best_estimator_

print('tune_rf() and tune_xgb() ready.')

tune_rf() and tune_xgb() ready.


### §5a-2 — Tune RF + XGB once per FS variant
Cached for reuse in §5b Stacker.

In [29]:
tuned_rf  = {fs: tune_rf (X_train[:, idx], y_train) for fs, idx in fs_methods.items()}
tuned_xgb = {fs: tune_xgb(X_train[:, idx], y_train) for fs, idx in fs_methods.items()}
print('Tuned RF + XGB cached for every FS variant.')

Tuned RF + XGB cached for every FS variant.


### §5a-3 — RandomForest (tuned) × 4 FS

In [30]:
ens_rf = []
for fs_name, fs_idx in fs_methods.items():
    Xtr = X_train[:, fs_idx]; Xte = X_test[:, fs_idx]
    r = evaluate(tuned_rf[fs_name], Xtr, Xte, y_train, y_test)
    r.update(model='RandomForest', category='Ensemble', fs=fs_name, n_features=len(fs_idx))
    ens_rf.append(r)
    print(f'  [Ensemble] RandomForest   | FS={fs_name:<8s} | '
          f'F1-macro={r["f1_macro"]:.4f}  Acc={r["acc"]:.4f}  F1={r["f1"]:.4f}  thr={r["thr"]:.2f}')

  [Ensemble] RandomForest   | FS=Full     | F1-macro=0.6989  Acc=0.8003  F1=0.5242  thr=0.56
  [Ensemble] RandomForest   | FS=RFE      | F1-macro=0.6809  Acc=0.7698  F1=0.5125  thr=0.50
  [Ensemble] RandomForest   | FS=Lasso-LR | F1-macro=0.6975  Acc=0.8025  F1=0.5193  thr=0.57
  [Ensemble] RandomForest   | FS=Forward  | F1-macro=0.6788  Acc=0.8053  F1=0.4772  thr=0.66


### §5a-4 — GradientBoosting (defaults) × 4 FS

In [ ]:
ens_gb = sweep_single_model(
    lambda: maybe_smote(GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=5,
        subsample=0.8, min_samples_leaf=10, random_state=RANDOM_STATE)),
    name='GradBoost', category='Ensemble'
)

### §5a-5 — XGBoost (tuned) × 4 FS

In [32]:
ens_xgb = []
for fs_name, fs_idx in fs_methods.items():
    Xtr = X_train[:, fs_idx]; Xte = X_test[:, fs_idx]
    r = evaluate(tuned_xgb[fs_name], Xtr, Xte, y_train, y_test)
    r.update(model='XGBoost', category='Ensemble', fs=fs_name, n_features=len(fs_idx))
    ens_xgb.append(r)
    print(f'  [Ensemble] XGBoost        | FS={fs_name:<8s} | '
          f'F1-macro={r["f1_macro"]:.4f}  Acc={r["acc"]:.4f}  F1={r["f1"]:.4f}  thr={r["thr"]:.2f}')

  [Ensemble] XGBoost        | FS=Full     | F1-macro=0.7001  Acc=0.8007  F1=0.5265  thr=0.54
  [Ensemble] XGBoost        | FS=RFE      | F1-macro=0.6865  Acc=0.7970  F1=0.5004  thr=0.56
  [Ensemble] XGBoost        | FS=Lasso-LR | F1-macro=0.6999  Acc=0.7988  F1=0.5276  thr=0.52
  [Ensemble] XGBoost        | FS=Forward  | F1-macro=0.6827  Acc=0.8023  F1=0.4879  thr=0.63


### §5a-6 — Classical ensembles table + save CSV

In [ ]:
ens_results = ens_rf + ens_gb + ens_xgb
df_ens = pd.DataFrame(ens_results)[
    ['category', 'model', 'fs', 'n_features', 'acc', 'f1', 'f1_macro', 'auc', 'thr']
]
df_ens.columns = ['Category', 'Model', 'FS', 'n_feat', 'Accuracy', 'F1', 'F1-macro', 'AUC', 'thr']
for c in ['Accuracy', 'F1', 'F1-macro', 'AUC']: df_ens[c] = df_ens[c].round(4)
df_ens['thr'] = df_ens['thr'].round(2)
print('=' * 88); print('  §5a CLASSICAL ENSEMBLES — 3 classifiers × 4 FS = 12 rows'); print('=' * 88)
print(df_ens.to_string(index=False))
save_csv(df_ens, RESULTS_DIR / f'{RUN_TAG}_classical_ensembles_results.csv')

### §5b — The Stacker (v3, 5 bases × 5 metas)

### §5b-1 — `cv_best_lambda()` helper for GA-MaxAccLogit meta

In [ ]:
def cv_best_lambda(X, y, lambda_grid, cv=3, n_gen=100, rs=RANDOM_STATE):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=rs)
    best_lam, best_score = lambda_grid[0], -np.inf
    for lam in lambda_grid:
        scores = []
        for tr, va in skf.split(X, y):
            mdl = maybe_smote(MaxAccLogit(lambda_reg=lam, n_gen=n_gen, scoring='f1_macro',
                                          random_state=rs)).fit(X[tr], y[tr])
            scores.append(f1_score(y[va], mdl.predict(X[va]), average='macro'))
        m = np.mean(scores)
        if m > best_score: best_score, best_lam = m, lam
    return best_lam

print('cv_best_lambda() ready.')

### §5b-2 — `run_stacker_v3()` — one call per FS variant

In [35]:
USE_AUGMENTED_META = False   # if True, meta sees (X + base_probs); else just base_probs

def run_stacker_v3(Xtr, Xte, ytr, yte, fs_name):
    print(f'\n  ── Stacker @ FS={fs_name}  (5 bases × 5 metas)'
          f'{"  | augmented meta input (X + base_probs)" if USE_AUGMENTED_META else ""} ──')
    base_models = {
        'RF'      : tuned_rf[fs_name],
        'GB'      : maybe_smote(GradientBoostingClassifier(
                       n_estimators=200, learning_rate=0.05, max_depth=5,
                       subsample=0.8, min_samples_leaf=10, random_state=RANDOM_STATE)),
        'XGB'     : tuned_xgb[fs_name],
        'KNN'     : maybe_smote(KNeighborsClassifier(n_neighbors=25, metric='manhattan')),
        'SVM'     : maybe_smote(SVC(C=1.0, kernel='rbf', gamma='scale',
                                    probability=True, random_state=RANDOM_STATE)),
    }
    oof_cols, test_cols = [], []
    for bname, bmodel in base_models.items():
        oof = cross_val_predict(bmodel, Xtr, ytr, cv=CV_FOLDS, method='predict_proba')[:, 1]
        oof_cols.append(oof)
        bmodel.fit(Xtr, ytr)
        test_cols.append(bmodel.predict_proba(Xte)[:, 1])
        print(f'    base {bname:<5s}  OOF AUC={roc_auc_score(ytr, oof):.4f}  '
              f'test AUC={roc_auc_score(yte, test_cols[-1]):.4f}')

    oof_probs_only  = np.column_stack(oof_cols)
    test_probs_only = np.column_stack(test_cols)

    if USE_AUGMENTED_META:
        # Stack raw features in front of the 5 base probabilities so meta sees both.
        oof_probs  = np.column_stack([Xtr, oof_probs_only])
        test_probs = np.column_stack([Xte, test_probs_only])
        print(f'    augmented meta input shape: train={oof_probs.shape}, test={test_probs.shape}'
              f'   (= {Xtr.shape[1]} raw features + 5 base probs)')
    else:
        oof_probs  = oof_probs_only
        test_probs = test_probs_only

    best_lam = cv_best_lambda(oof_probs, ytr,
                              lambda_grid=[0.001, 0.002, 0.005, 0.01],
                              cv=3, n_gen=100)
    meta_models = {
        'LR'  : maybe_smote(LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE)),
        'RF'  : maybe_smote(RandomForestClassifier(n_estimators=200, max_depth=5,
                                                   random_state=RANDOM_STATE)),
        'XGB' : maybe_smote(XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=3,
                                          random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0)),
        'MLP' : maybe_smote(MLPClassifier(hidden_layer_sizes=(64,32), activation='relu',
                                          alpha=0.001, max_iter=500, random_state=RANDOM_STATE)),
        'GA'  : maybe_smote(MaxAccLogit(lambda_reg=best_lam, n_gen=150,
                                        scoring='f1_macro', random_state=RANDOM_STATE)),
    }
    meta_scores = {}
    for mname, meta in meta_models.items():
        meta.fit(oof_probs, ytr)
        train_p = meta.predict_proba(oof_probs)[:, 1]
        proba   = meta.predict_proba(test_probs)[:, 1]
        if USE_GHOST:
            thr = ghostml.optimize_threshold_from_predictions(
                ytr, train_p, GHOST_THR, ThOpt_metrics='Kappa')
        else:
            thr = 0.5
        pred = (proba >= thr).astype(int)
        meta_scores[mname] = dict(
            acc=accuracy_score(yte, pred),
            f1 =f1_score(yte, pred),
            f1_macro=f1_score(yte, pred, average='macro'),
            auc=roc_auc_score(yte, proba),
            thr=thr,
        )
        s = meta_scores[mname]
        print(f'    meta {mname:<5s}  F1-macro={s["f1_macro"]:.4f}  '
              f'Acc={s["acc"]:.4f}  F1={s["f1"]:.4f}  AUC={s["auc"]:.4f}  thr={s["thr"]:.2f}')
    best_meta = max(meta_scores, key=lambda k: meta_scores[k]['f1_macro'])
    r = meta_scores[best_meta]
    r.update(model=f'Stacker ({best_meta})', category='Ensemble',
             fs=fs_name, n_features=Xtr.shape[1])
    print(f'    → winning meta: {best_meta}  (F1-macro={r["f1_macro"]:.4f})')
    return r

print('run_stacker_v3() ready.')

run_stacker_v3() ready.


### §5b-3 — Stacker on FS = Full

In [36]:
Xtr = X_train[:, fs_methods['Full']]; Xte = X_test[:, fs_methods['Full']]
stacker_full = run_stacker_v3(Xtr, Xte, y_train, y_test, 'Full')


  ── Stacker @ FS=Full  (5 bases × 5 metas) ──
    base RF     OOF AUC=0.7743  test AUC=0.7701
    base GB     OOF AUC=0.7682  test AUC=0.7682
    base XGB    OOF AUC=0.7720  test AUC=0.7654
    base KNN    OOF AUC=0.7314  test AUC=0.7319
    base SVM    OOF AUC=0.7557  test AUC=0.7535
    meta LR     F1-macro=0.7009  Acc=0.7997  F1=0.5290  AUC=0.7724  thr=0.63
    meta RF     F1-macro=0.6970  Acc=0.8002  F1=0.5202  AUC=0.7738  thr=0.66
    meta XGB    F1-macro=0.6992  Acc=0.8018  F1=0.5234  AUC=0.7731  thr=0.64
    meta MLP    F1-macro=0.7016  Acc=0.7983  F1=0.5317  AUC=0.7731  thr=0.62
    meta GA     F1-macro=0.7029  Acc=0.8003  F1=0.5328  AUC=0.7720  thr=0.56
    → winning meta: GA  (F1-macro=0.7029)


### §5b-4 — Stacker on FS = RFE

In [ ]:
Xtr = X_train[:, fs_methods['RFE']]; Xte = X_test[:, fs_methods['RFE']]
stacker_rfe = run_stacker_v3(Xtr, Xte, y_train, y_test, 'RFE')

### §5b-5 — Stacker on FS = Lasso-LR

In [ ]:
Xtr = X_train[:, fs_methods['Lasso-LR']]; Xte = X_test[:, fs_methods['Lasso-LR']]
stacker_lasso_lr = run_stacker_v3(Xtr, Xte, y_train, y_test, 'Lasso-LR')

### §5b-6 — Stacker on FS = Forward

In [ ]:
Xtr = X_train[:, fs_methods['Forward']]; Xte = X_test[:, fs_methods['Forward']]
stacker_forward = run_stacker_v3(Xtr, Xte, y_train, y_test, 'Forward')

### §5b-7 — Stacker results table + save CSV

In [ ]:
stacker_results = [stacker_full, stacker_rfe, stacker_lasso_lr, stacker_forward]
df_stack = pd.DataFrame(stacker_results)[
    ['category', 'model', 'fs', 'n_features', 'acc', 'f1', 'f1_macro', 'auc', 'thr']
]
df_stack.columns = ['Category', 'Model', 'FS', 'n_feat', 'Accuracy', 'F1', 'F1-macro', 'AUC', 'thr']
for c in ['Accuracy', 'F1', 'F1-macro', 'AUC']: df_stack[c] = df_stack[c].round(4)
df_stack['thr'] = df_stack['thr'].round(2)
print('=' * 88); print('  §5b STACKER — best meta per FS = 4 rows'); print('=' * 88)
print(df_stack.to_string(index=False))
save_csv(df_stack, RESULTS_DIR / f'{RUN_TAG}_stacker_results.csv')

## §6  Best model — every method on one page

Combined table from §3 + §5, sorted by F1-macro descending. Saved as `taiwan_v3_full_comparison.csv` plus a bar plot of the top 10.

### §6.1 — Combine + sort

In [41]:
df_all = pd.concat([df_base, df_ens, df_stack], ignore_index=True)
df_all = df_all.sort_values('F1-macro', ascending=False).reset_index(drop=True)
print('=' * 92); print('  §6 FINAL — every model × every FS, sorted by F1-macro'); print('=' * 92)
print(df_all.to_string(index=False))

  §6 FINAL — every model × every FS, sorted by F1-macro
Category         Model       FS  n_feat  Accuracy     F1  F1-macro    AUC  thr
Ensemble Stacker (MLP) Lasso-LR      21    0.7983 0.5371    0.7041 0.7734 0.61
Ensemble  Stacker (GA)     Full      23    0.8003 0.5328    0.7029 0.7720 0.56
Ensemble       XGBoost     Full      23    0.8007 0.5265    0.7001 0.7654 0.54
Ensemble       XGBoost Lasso-LR      21    0.7988 0.5276    0.6999 0.7655 0.52
Ensemble  RandomForest     Full      23    0.8003 0.5242    0.6989 0.7701 0.56
Ensemble  RandomForest Lasso-LR      21    0.8025 0.5193    0.6975 0.7709 0.57
Ensemble     GradBoost     Full      23    0.7962 0.5236    0.6970 0.7682 0.46
    Base       SVM-RBF Lasso-LR      21    0.8028 0.5173    0.6967 0.7529 0.75
Ensemble     GradBoost Lasso-LR      21    0.7902 0.5240    0.6947 0.7661 0.44
    Base  DecisionTree Lasso-LR      21    0.8057 0.4996    0.6895 0.7445 0.61
    Base       SVM-RBF     Full      23    0.8013 0.5000    0.6880 0.7535 0

### §6.2 — Save final CSV

In [ ]:
save_csv(df_all, RESULTS_DIR / f'{RUN_TAG}_full_comparison.csv')

### §6.3 — Bar plot of top 10

In [ ]:
top = df_all.head(10).copy()
labels = [f'{m}\n@{fs}' for m, fs in zip(top['Model'], top['FS'])]
colors = {'Base': '#3498db', 'Ensemble': '#27ae60'}
bar_colors = [colors[c] for c in top['Category']]

fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(range(len(top)), top['F1-macro'], color=bar_colors)
ax.set_yticks(range(len(top))); ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel('F1-macro (higher is better)')
ax.set_title('Taiwan Credit Card Default (v3 + FS) — Top 10 by F1-macro')
for i, v in enumerate(top['F1-macro']):
    ax.text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=9)
ax.grid(axis='x', alpha=0.3, ls='--')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=colors[k], label=k) for k in colors], loc='lower right')
plt.tight_layout()
plot_path = RESULTS_DIR / f'{RUN_TAG}_top10_f1macro.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {plot_path}')

### §6.4 — Winner declaration

In [ ]:
winner = df_all.iloc[0]
print('=' * 70); print('  WINNER on Taiwan Credit Card Default (by F1-macro):'); print('=' * 70)
print(f'  Model:    {winner["Model"]}')
print(f'  Category: {winner["Category"]}')
print(f'  FS:       {winner["FS"]}  ({winner["n_feat"]} features)')
print(f'  Accuracy: {winner["Accuracy"]:.4f}')
print(f'  F1:       {winner["F1"]:.4f}')
print(f'  F1-macro: {winner["F1-macro"]:.4f}')
print(f'  AUC:      {winner["AUC"]:.4f}')
print(f'  thr:      {winner["thr"]:.2f}')

stackers = df_all[df_all['Model'].str.startswith('Stacker')]
non_stacker = df_all[~df_all['Model'].str.startswith('Stacker')]
if len(stackers) and len(non_stacker):
    best_stacker = stackers.iloc[0]
    best_non_stacker = non_stacker.iloc[0]
    margin = best_stacker['F1-macro'] - best_non_stacker['F1-macro']
    print(f'\n  Best Stacker:     {best_stacker["Model"]} @ {best_stacker["FS"]}  F1-macro={best_stacker["F1-macro"]:.4f}')
    print(f'  Best non-stacker: {best_non_stacker["Model"]} @ {best_non_stacker["FS"]}  F1-macro={best_non_stacker["F1-macro"]:.4f}')
    print(f'  Stacker margin:   {margin:+.4f} F1-macro')

### §6.5 — Recap of every artefact saved

In [45]:
print('=' * 70); print('  Files saved by this notebook'); print('=' * 70)
for d in [SUBSETS_DIR, RESULTS_DIR]:
    for p in sorted(d.glob(f'{RUN_TAG}*')):
        size = p.stat().st_size
        print(f'  {p}  ({size:,} bytes)')

  Files saved by this notebook
  /content/drive/MyDrive/CreditRiskApp/tier2_predictive_models/fs_subsets/taiwan_v3_FS_Forward.csv  (6,814,934 bytes)
  /content/drive/MyDrive/CreditRiskApp/tier2_predictive_models/fs_subsets/taiwan_v3_FS_Forward_features.csv  (81 bytes)
  /content/drive/MyDrive/CreditRiskApp/tier2_predictive_models/fs_subsets/taiwan_v3_FS_Full.csv  (13,903,528 bytes)
  /content/drive/MyDrive/CreditRiskApp/tier2_predictive_models/fs_subsets/taiwan_v3_FS_Full_features.csv  (154 bytes)
  /content/drive/MyDrive/CreditRiskApp/tier2_predictive_models/fs_subsets/taiwan_v3_FS_Lasso_LR.csv  (12,711,272 bytes)
  /content/drive/MyDrive/CreditRiskApp/tier2_predictive_models/fs_subsets/taiwan_v3_FS_Lasso_LR_features.csv  (141 bytes)
  /content/drive/MyDrive/CreditRiskApp/tier2_predictive_models/fs_subsets/taiwan_v3_FS_RFE.csv  (6,806,774 bytes)
  /content/drive/MyDrive/CreditRiskApp/tier2_predictive_models/fs_subsets/taiwan_v3_FS_RFE_features.csv  (83 bytes)
  /content/drive/MyDrive/

## §7  Ten-measure evaluation (Chi et al., 2019)

This section adds the **ten credit-scoring measures** — Accuracy, AUC, Type I
error, Type II error, EMCC, G-Mean, Discriminant Power, F-Score, Kappa and
Youden's index — for every model and feature-selection variant.

The earlier model cells (§3, §5) store only Acc / F1 / F1-macro / AUC and
discard the predictions, so each model is **re-fitted here** to recover the full
confusion matrix. Every existing cell above is left unchanged.

### §7.1 — Ten-measure helper `ten_measures()`

In [46]:
# §7.1 -- Ten-measure suite (Chi et al., 2019).  1 = good = positive.
from sklearn.metrics import confusion_matrix

def ten_measures(y_true, y_pred, proba, cost_bad=5.0, cost_good=1.0):
    """Ten credit-scoring measures from labels + predicted probability."""
    y_true = np.asarray(y_true, int)
    y_pred = np.asarray(y_pred, int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    e = 1e-9
    total = tp + tn + fp + fn
    sens = tp / (tp + fn + e)
    spec = tn / (tn + fp + e)
    prec = tp / (tp + fp + e)
    type1 = fn / (tp + fn + e)
    type2 = fp / (fp + tn + e)
    pi_good = (tp + fn) / (total + e)
    pi_bad  = (tn + fp) / (total + e)
    sc  = min(max(sens, e), 1 - e)
    spc = min(max(spec, e), 1 - e)
    return dict(
        Accuracy = (tp + tn) / (total + e),
        AUC      = roc_auc_score(y_true, proba),
        TypeI    = type1,
        TypeII   = type2,
        EMCC     = cost_bad * pi_bad * type2 + cost_good * pi_good * type1,
        GMean    = np.sqrt(max(sens, 0.0) * max(spec, 0.0)),
        DP       = (np.sqrt(3) / np.pi) * (np.log(sc / (1 - sc))
                                           + np.log(spc / (1 - spc))),
        FScore   = 2 * prec * sens / (prec + sens + e),
        Kappa    = cohen_kappa_score(y_true, y_pred),
        Youden   = sens + spec - 1.0,
    )

print('ten_measures() ready -- Accuracy, AUC, Type I, Type II, EMCC, '
      'G-Mean, DP, F-Score, Kappa, Youden.')

ten_measures() ready -- Accuracy, AUC, Type I, Type II, EMCC, G-Mean, DP, F-Score, Kappa, Youden.


### §7.2 — `evaluate_ten()` and `sweep_ten()` (mirror `evaluate` / `sweep_single_model`)

In [47]:
# §7.2 -- mirror evaluate() / sweep_single_model() from §2, returning ten measures.
def evaluate_ten(model, X_tr, X_te, y_tr, y_te):
    """Fit on (X_tr, y_tr); predict on X_te; return the ten measures + thr.
    Same fit/predict and GHOST threshold logic as evaluate()."""
    model.fit(X_tr, y_tr)
    proba_test = model.predict_proba(X_te)[:, 1]
    if USE_GHOST:
        proba_train = model.predict_proba(X_tr)[:, 1]
        thr = ghostml.optimize_threshold_from_predictions(
            y_tr, proba_train, GHOST_THR, ThOpt_metrics='Kappa')
    else:
        thr = 0.5
    pred = (proba_test >= thr).astype(int)
    r = ten_measures(y_te, pred, proba_test)
    r['thr'] = thr
    return r

def sweep_ten(model_factory, name, category):
    """Run one model_factory across all 4 FS variants -- ten-measure version."""
    rows = []
    for fs_name, fs_idx in fs_methods.items():
        Xtr = X_train[:, fs_idx]; Xte = X_test[:, fs_idx]
        r = evaluate_ten(model_factory(), Xtr, Xte, y_train, y_test)
        r.update(model=name, category=category, fs=fs_name, n_features=len(fs_idx))
        rows.append(r)
        print(f'  [{category}] {name:<14s} | FS={fs_name:<8s} | '
              f'F-Score={r["FScore"]:.4f}  G-Mean={r["GMean"]:.4f}  '
              f'AUC={r["AUC"]:.4f}  Kappa={r["Kappa"]:.4f}')
    return rows

print('evaluate_ten() and sweep_ten() ready.')

evaluate_ten() and sweep_ten() ready.


### §7.3 — Base models × 4 FS (ten measures)

In [48]:
# §7.3 -- Base models, ten measures (same factories as §3).
base_lr_10 = sweep_ten(
    lambda: maybe_smote(LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE)),
    name='LogisticReg', category='Base')
base_knn_10 = sweep_ten(
    lambda: maybe_smote(KNeighborsClassifier(n_neighbors=25, metric='manhattan')),
    name='KNN', category='Base')
base_svm_10 = sweep_ten(
    lambda: maybe_smote(SVC(C=1.0, kernel='rbf', gamma='scale',
                            probability=True, random_state=RANDOM_STATE)),
    name='SVM-RBF', category='Base')
base_dt_10 = sweep_ten(
    lambda: maybe_smote(DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)),
    name='DecisionTree', category='Base')

  [Base] LogisticReg    | FS=Full     | F-Score=0.4933  G-Mean=0.6257  AUC=0.7105  Kappa=0.3757
  [Base] LogisticReg    | FS=RFE      | F-Score=0.4783  G-Mean=0.6143  AUC=0.7066  Kappa=0.3579
  [Base] LogisticReg    | FS=Lasso-LR | F-Score=0.4935  G-Mean=0.6254  AUC=0.7108  Kappa=0.3764
  [Base] LogisticReg    | FS=Forward  | F-Score=0.4975  G-Mean=0.6468  AUC=0.7006  Kappa=0.3616
  [Base] KNN            | FS=Full     | F-Score=0.4836  G-Mean=0.6399  AUC=0.7319  Kappa=0.3398
  [Base] KNN            | FS=RFE      | F-Score=0.4635  G-Mean=0.6167  AUC=0.7264  Kappa=0.3237
  [Base] KNN            | FS=Lasso-LR | F-Score=0.4868  G-Mean=0.6446  AUC=0.7327  Kappa=0.3413
  [Base] KNN            | FS=Forward  | F-Score=0.4835  G-Mean=0.6378  AUC=0.7238  Kappa=0.3422
  [Base] SVM-RBF        | FS=Full     | F-Score=0.5000  G-Mean=0.6363  AUC=0.7535  Kappa=0.3780
  [Base] SVM-RBF        | FS=RFE      | F-Score=0.4792  G-Mean=0.6153  AUC=0.7458  Kappa=0.3586
  [Base] SVM-RBF        | FS=Lasso-LR | 

### §7.4 — Ensemble models × 4 FS (ten measures)

In [ ]:
# §7.4 -- Ensemble models, ten measures (RF and XGB reuse the per-FS tuned models).
ens_rf_10 = []
for fs_name, fs_idx in fs_methods.items():
    Xtr = X_train[:, fs_idx]; Xte = X_test[:, fs_idx]
    r = evaluate_ten(tuned_rf[fs_name], Xtr, Xte, y_train, y_test)
    r.update(model='RandomForest', category='Ensemble', fs=fs_name, n_features=len(fs_idx))
    ens_rf_10.append(r)
    print(f'  [Ensemble] RandomForest   | FS={fs_name:<8s} | '
          f'F-Score={r["FScore"]:.4f}  G-Mean={r["GMean"]:.4f}  '
          f'AUC={r["AUC"]:.4f}  Kappa={r["Kappa"]:.4f}')

ens_gb_10 = sweep_ten(
    lambda: maybe_smote(GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=5,
        subsample=0.8, min_samples_leaf=10, random_state=RANDOM_STATE)),
    name='GradBoost', category='Ensemble')

ens_xgb_10 = []
for fs_name, fs_idx in fs_methods.items():
    Xtr = X_train[:, fs_idx]; Xte = X_test[:, fs_idx]
    r = evaluate_ten(tuned_xgb[fs_name], Xtr, Xte, y_train, y_test)
    r.update(model='XGBoost', category='Ensemble', fs=fs_name, n_features=len(fs_idx))
    ens_xgb_10.append(r)
    print(f'  [Ensemble] XGBoost        | FS={fs_name:<8s} | '
          f'F-Score={r["FScore"]:.4f}  G-Mean={r["GMean"]:.4f}  '
          f'AUC={r["AUC"]:.4f}  Kappa={r["Kappa"]:.4f}')

### §7.5 — Stacker × 4 FS (`run_stacker_v3_ten`, ten-measure copy of `run_stacker_v3`)

In [ ]:
# §7.5 -- ten-measure copy of run_stacker_v3 (cell in §5b-2). Same 5 bases,
# 5 metas, same GHOST meta-threshold and winning-meta selection (by F1-macro).
def run_stacker_v3_ten(Xtr, Xte, ytr, yte, fs_name):
    print(f'\n  -- Stacker @ FS={fs_name}  (ten-measure eval) --')
    base_models = {
        'RF'  : tuned_rf[fs_name],
        'GB'  : maybe_smote(GradientBoostingClassifier(
                   n_estimators=200, learning_rate=0.05, max_depth=5,
                   subsample=0.8, min_samples_leaf=10, random_state=RANDOM_STATE)),
        'XGB' : tuned_xgb[fs_name],
        'KNN' : maybe_smote(KNeighborsClassifier(n_neighbors=25, metric='manhattan')),
        'SVM' : maybe_smote(SVC(C=1.0, kernel='rbf', gamma='scale',
                                probability=True, random_state=RANDOM_STATE)),
    }
    oof_cols, test_cols = [], []
    for bname, bmodel in base_models.items():
        oof = cross_val_predict(bmodel, Xtr, ytr, cv=CV_FOLDS,
                                method='predict_proba')[:, 1]
        oof_cols.append(oof)
        bmodel.fit(Xtr, ytr)
        test_cols.append(bmodel.predict_proba(Xte)[:, 1])
    oof_probs  = np.column_stack(oof_cols)
    test_probs = np.column_stack(test_cols)

    best_lam = cv_best_lambda(oof_probs, ytr,
                              lambda_grid=[0.001, 0.002, 0.005, 0.01],
                              cv=3, n_gen=100)
    meta_models = {
        'LR'  : maybe_smote(LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE)),
        'RF'  : maybe_smote(RandomForestClassifier(n_estimators=200, max_depth=5,
                                                   random_state=RANDOM_STATE)),
        'XGB' : maybe_smote(XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=3,
                                          random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0)),
        'MLP' : maybe_smote(MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu',
                                          alpha=0.001, max_iter=500, random_state=RANDOM_STATE)),
        'GA'  : maybe_smote(MaxAccLogit(lambda_reg=best_lam, n_gen=150,
                                        scoring='f1_macro', random_state=RANDOM_STATE)),
    }
    meta_eval = {}
    for mname, meta in meta_models.items():
        meta.fit(oof_probs, ytr)
        train_p = meta.predict_proba(oof_probs)[:, 1]
        proba   = meta.predict_proba(test_probs)[:, 1]
        if USE_GHOST:
            thr = ghostml.optimize_threshold_from_predictions(
                ytr, train_p, GHOST_THR, ThOpt_metrics='Kappa')
        else:
            thr = 0.5
        pred = (proba >= thr).astype(int)
        m = ten_measures(yte, pred, proba)
        m['f1_macro'] = f1_score(yte, pred, average='macro')
        m['thr'] = thr
        meta_eval[mname] = m
        print(f'    meta {mname:<5s}  F-Score={m["FScore"]:.4f}  G-Mean={m["GMean"]:.4f}  '
              f'AUC={m["AUC"]:.4f}  Kappa={m["Kappa"]:.4f}  thr={thr:.2f}')
    best_meta = max(meta_eval, key=lambda k: meta_eval[k]['f1_macro'])
    r = {k: v for k, v in meta_eval[best_meta].items() if k != 'f1_macro'}
    r.update(model=f'Stacker ({best_meta})', category='Ensemble',
             fs=fs_name, n_features=Xtr.shape[1])
    print(f'    -> winning meta: {best_meta}  (F-Score={r["FScore"]:.4f})')
    return r

print('run_stacker_v3_ten() ready.')

In [ ]:
# §7.5 -- run the ten-measure stacker once per FS variant.
stk_full_10  = run_stacker_v3_ten(X_train[:, fs_methods['Full']],
                                  X_test[:,  fs_methods['Full']],     y_train, y_test, 'Full')
stk_rfe_10   = run_stacker_v3_ten(X_train[:, fs_methods['RFE']],
                                  X_test[:,  fs_methods['RFE']],      y_train, y_test, 'RFE')
stk_lasso_10 = run_stacker_v3_ten(X_train[:, fs_methods['Lasso-LR']],
                                  X_test[:,  fs_methods['Lasso-LR']], y_train, y_test, 'Lasso-LR')
stk_fwd_10   = run_stacker_v3_ten(X_train[:, fs_methods['Forward']],
                                  X_test[:,  fs_methods['Forward']],  y_train, y_test, 'Forward')

### §7.6 — Ten-measure results table + save CSV

In [52]:
# §7.6 -- combine, sort, display and save the ten-measure results.
all_ten = (base_lr_10 + base_knn_10 + base_svm_10 + base_dt_10
           + ens_rf_10 + ens_gb_10 + ens_xgb_10
           + [stk_full_10, stk_rfe_10, stk_lasso_10, stk_fwd_10])

df_ten = pd.DataFrame(all_ten)[
    ['category', 'model', 'fs', 'n_features',
     'Accuracy', 'AUC', 'TypeI', 'TypeII', 'EMCC',
     'GMean', 'DP', 'FScore', 'Kappa', 'Youden', 'thr']
]
df_ten.columns = ['Category', 'Model', 'FS', 'n_feat',
                  'Accuracy', 'AUC', 'Type I', 'Type II', 'EMCC',
                  'G-Mean', 'DP', 'F-Score', 'Kappa', 'Youden', 'thr']
for c in ['Accuracy', 'AUC', 'Type I', 'Type II', 'EMCC',
          'G-Mean', 'DP', 'F-Score', 'Kappa', 'Youden']:
    df_ten[c] = df_ten[c].round(4)
df_ten['thr'] = df_ten['thr'].round(2)
df_ten = df_ten.sort_values('F-Score', ascending=False).reset_index(drop=True)

print('=' * 110)
print('  §7 TEN-MEASURE EVALUATION -- every model x every FS (sorted by F-Score)')
print('=' * 110)
print(df_ten.to_string(index=False))
print('\nHigher is better EXCEPT Type I, Type II, EMCC (lower is better).')
save_csv(df_ten, RESULTS_DIR / f'{RUN_TAG}_ten_measure_results.csv')

  §7 TEN-MEASURE EVALUATION -- every model x every FS (sorted by F-Score)
Category        Model       FS  n_feat  Accuracy    AUC  Type I  Type II   EMCC  G-Mean     DP  F-Score  Kappa  Youden  thr
Ensemble Stacker (GA)     Full      23    0.7988 0.7720  0.4740   0.1237 0.5865  0.6789 1.1369   0.5363 0.4079  0.4023 0.55
Ensemble Stacker (GA) Lasso-LR      21    0.7972 0.7712  0.4702   0.1269 0.5982  0.6801 1.1290   0.5360 0.4063  0.4029 0.53
Ensemble RandomForest     Full      23    0.7978 0.7701  0.4891   0.1207 0.5782  0.6703 1.1190   0.5278 0.3994  0.3902 0.54
Ensemble      XGBoost Lasso-LR      21    0.7988 0.7655  0.4921   0.1186 0.5705  0.6691 1.1235   0.5276 0.4000  0.3894 0.52
Ensemble      XGBoost     Full      23    0.8007 0.7654  0.4989   0.1143 0.5553  0.6662 1.1315   0.5265 0.4007  0.3869 0.54
Ensemble    GradBoost     Full      23    0.7937 0.7682  0.4823   0.1280 0.6050  0.6719 1.0971   0.5260 0.3942  0.3897 0.45
Ensemble    GradBoost Lasso-LR      21    0.7930 0.7661  0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>